# Pipeline de Entrenamiento: Modelo de Riesgo de Prestamos

Este notebook implementa el flujo completo de entrenamiento para la prediccion de impagos utilizando XGBoost. El proceso consume datos procesados desde **Azure Database for PostgreSQL** (gestionados con **dbt**), optimiza hiperparametros con **Optuna** y registra experimentos en **MLflow**.

---

## 1. Configuracion y Conexion a Azure
Cargamos las credenciales desde el archivo `.env` para asegurar una conexion protegida a la base de datos en la nube.

In [5]:
import os
import pandas as pd
import optuna
import mlflow
from dotenv import load_dotenv
from sqlalchemy import create_engine
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import urllib.parse

# Carga de variables de entorno desde la raiz del proyecto
load_dotenv('../.env')

# Parametros de conexion obtenidos de forma segura
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")

if not all([DB_USER, DB_PASS, DB_HOST, DB_NAME]):
    raise ValueError("Error: No se pudieron cargar las credenciales del archivo .env")

# Para Azure PostgreSQL, el usuario debe incluir el nombre del servidor
# DB_USER = f"{DB_USER}@lending-club-db2"  # Comentado, intentar sin @server

# Codificar la contraseña para evitar problemas con caracteres especiales como @
DB_PASS_encoded = urllib.parse.quote(DB_PASS, safe='')

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASS_encoded}@{DB_HOST}:5432/{DB_NAME}?sslmode=require"
engine = create_engine(DATABASE_URL)

# Configuracion del experimento en MLflow
mlflow.set_experiment("Lending_Club_Risk_v1")

print(f"Conectando a Azure PostgreSQL: {DB_HOST}...")
# Consumo de la vista generada por dbt en la capa de analytics
query = "SELECT * FROM ml_input"
df = pd.read_sql(query, engine)
print("Conexion exitosa")
print(f"Dimensiones del dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
display(df.head())

Conectando a Azure PostgreSQL: lending-club-db2.postgres.database.azure.com...
Conexion exitosa
Dimensiones del dataset cargado: 1860764 filas, 7 columnas


,loan_id,loan_amnt,int_rate,annual_inc,dti,installment,target
0,841875,4000.0,13.49,29004.0,19.65,92.02,0
1,849438,35000.0,16.89,85000.0,17.60,867.78,0
2,856059,25000.0,13.49,108900.0,6.49,575.12,1
3,1025314,6000.0,9.91,57600.0,7.98,193.35,0
4,829150,5400.0,5.99,120000.0,4.46,164.26,0


## 2. Preparacion del Dataset
Se evalua el desbalanceo de clases y se dividen los datos en conjuntos de entrenamiento y prueba, manteniendo la proporcion de la variable objetivo.

In [6]:
print("Distribucion de clases en la variable objetivo (target)")
print(df['target'].value_counts())
print(f"Clase 0 (Pagado): {(df['target'] == 0).mean() * 100:.2f}%")
print(f"Clase 1 (Impago): {(df['target'] == 1).mean() * 100:.2f}%")

# Separacion de caracteristicas y etiquetas
X = df.drop(['loan_id', 'target'], axis=1)
y = df['target']

# Division estratificada del dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Registros de entrenamiento: {X_train.shape[0]}")
print(f"Registros de prueba: {X_test.shape[0]}")

Distribucion de clases en la variable objetivo (target)
target
0    1497783
1     362981
Name: count, dtype: int64
Clase 0 (Pagado): 80.49%
Clase 1 (Impago): 19.51%
Registros de entrenamiento: 1488611
Registros de prueba: 372153


## 3. Optimizacion de Hiperparametros con Optuna
Se define una funcion objetivo para buscar los mejores parametros de XGBoost que maximicen el F1-Score. Cada iteracion se registra de forma anidada en MLflow.

In [7]:
def objective(trial):
    # Registro anidado para cada prueba de Optuna
    with mlflow.start_run(nested=True):
        # Espacio de busqueda de hiperparametros
        param = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 200),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
            'scale_pos_weight': trial.suggest_float(
                'scale_pos_weight',
                1,
                (y_train == 0).sum() / (y_train == 1).sum()
            ),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'gamma': trial.suggest_float('gamma', 0, 5),
            'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
            'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
            'min_child_weight': trial.suggest_float('min_child_weight', 1, 10),
            'random_state': 42,
            'eval_metric': 'logloss',
            'tree_method': 'hist',
            'device': 'cuda', 
            'use_label_encoder': False,
        }

        # Entrenamiento del modelo
        model = XGBClassifier(**param)
        model.fit(X_train, y_train)

        # Evaluacion de metricas
        preds = model.predict(X_test)
        f1 = f1_score(y_test, preds)
        recall = recall_score(y_test, preds)
        acc = accuracy_score(y_test, preds)

        # Registro en MLflow
        mlflow.log_params(param)
        mlflow.log_metric('f1_score', f1)
        mlflow.log_metric('recall', recall)
        mlflow.log_metric('accuracy', acc)

        return f1

# Inicio del experimento de optimizacion
with mlflow.start_run(run_name='Optuna_Optimization_XGB') as parent_run:
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=100)

    print('Mejores hiperparametros encontrados:')
    print(study.best_params)
    print(f'Mejor F1 alcanzado: {study.best_value:.4f}')

    # Entrenamiento del mejor modelo final
    best_model = XGBClassifier(**study.best_params, tree_method='hist', device='cuda', use_label_encoder=False)
    best_model.fit(X_train, y_train)

    # Evaluacion final en el test set
    final_preds = best_model.predict(X_test)
    final_f1 = f1_score(y_test, final_preds)
    final_recall = recall_score(y_test, final_preds)
    final_acc = accuracy_score(y_test, final_preds)

    # Registro de resultados finales
    mlflow.log_params(study.best_params)
    mlflow.log_metric('best_f1_score', final_f1)
    mlflow.log_metric('best_recall', final_recall)
    mlflow.log_metric('best_accuracy', final_acc)

    print('\nPerformance final del modelo:')
    print(f'Accuracy: {final_acc:.4f}')
    print(f'Recall:  {final_recall:.4f}')
    print(f'F1 Score: {final_f1:.4f}')
    print('\nMatriz de Confusion:')
    print(confusion_matrix(y_test, final_preds))

    # Persistencia del modelo en MLflow
    mlflow.sklearn.log_model(best_model, 'best_model_lending_club')

best_params = study.best_params
best_model_final = best_model

[I 2026-04-19 02:03:08,983] A new study created in memory with name: no-name-7a9d9f0b-97ee-45a7-a3bb-b8b9fa08f3ec
c:\Users\David\Documents\PBI\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [02:03:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\David\Documents\PBI\.venv\Lib\site-packages\xgboost\core.py:751: UserWarning: [02:03:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
[I 2026-04-19 02:03:10,818] Tria

Mejores hiperparametros encontrados:
{'n_estimators': 128, 'max_depth': 7, 'learning_rate': 0.10228108508049931, 'scale_pos_weight': 3.831754637622857, 'subsample': 0.9840106367807223, 'colsample_bytree': 0.9918675911625324, 'gamma': 2.0905821027526525, 'reg_alpha': 4.099860811630099, 'reg_lambda': 0.8286305232544637, 'min_child_weight': 1.9475389154587095}
Mejor F1 alcanzado: 0.4156


c:\Users\David\Documents\PBI\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [02:04:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026/04/19 02:04:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 02:04:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Performance final del modelo:
Accuracy: 0.6571
Recall:  0.6233
F1 Score: 0.4150

Matriz de Confusion:
[[199309 100248]
 [ 27345  45251]]


2026/04/19 02:05:03 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


## 4. Ajuste del Umbral de Decision y Inferencia Universal
Se analizan diferentes umbrales para optimizar la sensibilidad del modelo y se generan predicciones para TODO el universo de clientes (incluyendo los actuales) para su visualizacion en Power BI.

In [8]:
# Rango de umbrales a evaluar
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50]
probs = best_model_final.predict_proba(X_test)[:, 1]

with mlflow.start_run(run_name='Threshold_Tuning') as threshold_run:
    best_f1 = 0.0
    best_threshold = thresholds[0]

    print(f"{'Umbral':<10}{'Accuracy':<10}{'Recall':<10}{'F1':<10}")
    print('-' * 40)

    for thresh in thresholds:
        preds = (probs >= thresh).astype(int)
        acc = accuracy_score(y_test, preds)
        rec = recall_score(y_test, preds)
        f1 = f1_score(y_test, preds)

        print(f"{thresh:<10.2f}{acc:<10.4f}{rec:<10.4f}{f1:<10.4f}")

        with mlflow.start_run(run_name=f'Thresh_{thresh}', nested=True):
            mlflow.log_param('threshold', thresh)
            mlflow.log_metric('accuracy', acc)
            mlflow.log_metric('recall', rec)
            mlflow.log_metric('f1_score', f1)

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = thresh

    mlflow.log_param('best_threshold', best_threshold)
    print(f"\nUmbral optimo seleccionado: {best_threshold}")

    # Generacion de scores para todo el universo de clientes (Inferencia)
    print("Cargando datos de todos los clientes para inferencia...")
    df_all = pd.read_sql("SELECT * FROM v_inference_input", engine)
    
    # Preparamos los datos para el modelo (mismas columnas que en el entrenamiento)
    # Asegurarse de que X_all tenga exactamente las mismas columnas que X_train
    X_all = df_all.drop(['loan_id', 'loan_status'], axis=1)
    
    print("Generando predicciones de riesgo universales...")
    full_probs = best_model_final.predict_proba(X_all)[:, 1]
    df_all['risk_score'] = full_probs
    df_all['final_prediction'] = (full_probs >= best_threshold).astype(int)

    # Carga de predicciones a Azure PostgreSQL
    print("Exportando predicciones a la tabla fact_predictions en Azure...")
    df_all[['loan_id', 'risk_score', 'final_prediction']].to_sql(
        'fact_predictions',
        engine,
        if_exists='replace',
        index=False
    )

    print('¡Proceso finalizado! Predicciones cargadas en fact_predictions.')

Umbral    Accuracy  Recall    F1        
----------------------------------------
0.30      0.3903    0.9246    0.3717    
0.35      0.4544    0.8778    0.3856    
0.40      0.5257    0.8106    0.4000    
0.45      0.5942    0.7251    0.4107    
0.50      0.6571    0.6233    0.4150    

Umbral optimo seleccionado: 0.5
Cargando datos de todos los clientes para inferencia...
Generando predicciones de riesgo universales...
Exportando predicciones a la tabla fact_predictions en Azure...
¡Proceso finalizado! Predicciones cargadas en fact_predictions.
